In [2]:
%pip install transformers torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "openai-community/gpt2"

hf_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

hf_model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
class GPT2Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config["n_head"]
        self.head_dim = config["n_embd"] // self.num_heads

        self.c_attn = nn.Linear(config["n_embd"], 3 * config["n_embd"])
        self.c_proj = nn.Linear(config["n_embd"], config["n_embd"])

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config["n_positions"], config["n_positions"]))
            .view(1, 1, config["n_positions"], config["n_positions"])
        )

    def forward(self, x):
        batch, seq_len, emb = x.shape

        q, k, v = self.c_attn(x).split(emb, dim=2)

        def split_heads(t):
            return t.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        q = split_heads(q)
        k = split_heads(k)
        v = split_heads(v)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        scores = scores.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(batch, seq_len, emb)
        return self.c_proj(out)


class GPT2MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config["n_embd"], 4 * config["n_embd"])
        self.c_proj = nn.Linear(4 * config["n_embd"], config["n_embd"])

    def forward(self, x):
        return self.c_proj(F.gelu(self.c_fc(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config["n_embd"], eps=config["layer_norm_epsilon"])
        self.attn = GPT2Attention(config)
        self.ln_2 = nn.LayerNorm(config["n_embd"], eps=config["layer_norm_epsilon"])
        self.mlp = GPT2MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MyGPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.wte = nn.Embedding(config["vocab_size"], config["n_embd"])
        self.wpe = nn.Embedding(config["n_positions"], config["n_embd"])
        self.drop = nn.Dropout(config["embd_pdrop"])

        self.h = nn.ModuleList([GPT2Block(config) for _ in range(config["n_layer"])])
        self.ln_f = nn.LayerNorm(config["n_embd"], eps=config["layer_norm_epsilon"])

    def forward(self, input_ids):
        batch, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        x = self.drop(self.wte(input_ids) + self.wpe(positions))

        for block in self.h:
            x = block(x)

        x = self.ln_f(x)
        logits = F.linear(x, self.wte.weight)
        return logits

In [4]:
config = {
    "vocab_size": hf_model.config.vocab_size,
    "n_positions": hf_model.config.n_positions,
    "n_embd": hf_model.config.n_embd,
    "n_layer": hf_model.config.n_layer,
    "n_head": hf_model.config.n_head,
    "layer_norm_epsilon": hf_model.config.layer_norm_epsilon,
    "embd_pdrop": hf_model.config.embd_pdrop,
}

my_model = MyGPT2(config).to(device)

print(config)

{'vocab_size': 50257, 'n_positions': 1024, 'n_embd': 768, 'n_layer': 12, 'n_head': 12, 'layer_norm_epsilon': 1e-05, 'embd_pdrop': 0.1}


In [5]:
with torch.no_grad():
    my_model.wte.weight.copy_(hf_model.transformer.wte.weight)
    my_model.wpe.weight.copy_(hf_model.transformer.wpe.weight)
    my_model.ln_f.weight.copy_(hf_model.transformer.ln_f.weight)
    my_model.ln_f.bias.copy_(hf_model.transformer.ln_f.bias)

    for my_block, hf_block in zip(my_model.h, hf_model.transformer.h):
        my_block.ln_1.weight.copy_(hf_block.ln_1.weight)
        my_block.ln_1.bias.copy_(hf_block.ln_1.bias)

        my_block.ln_2.weight.copy_(hf_block.ln_2.weight)
        my_block.ln_2.bias.copy_(hf_block.ln_2.bias)

        # Attention Q, K, V tek bir katmanda
        my_block.attn.c_attn.weight.copy_(hf_block.attn.c_attn.weight.T)
        my_block.attn.c_attn.bias.copy_(hf_block.attn.c_attn.bias)

        my_block.attn.c_proj.weight.copy_(hf_block.attn.c_proj.weight.T)
        my_block.attn.c_proj.bias.copy_(hf_block.attn.c_proj.bias)

        # MLP
        my_block.mlp.c_fc.weight.copy_(hf_block.mlp.c_fc.weight.T)
        my_block.mlp.c_fc.bias.copy_(hf_block.mlp.c_fc.bias)

        my_block.mlp.c_proj.weight.copy_(hf_block.mlp.c_proj.weight.T)
        my_block.mlp.c_proj.bias.copy_(hf_block.mlp.c_proj.bias)

my_model.eval()
for name, hf_weight, my_weight in [
    ("c_attn", hf_block.attn.c_attn.weight, my_block.attn.c_attn.weight),
    ("attn.c_proj", hf_block.attn.c_proj.weight, my_block.attn.c_proj.weight),
    ("mlp.c_fc", hf_block.mlp.c_fc.weight, my_block.mlp.c_fc.weight),
    ("mlp.c_proj", hf_block.mlp.c_proj.weight, my_block.mlp.c_proj.weight),
]:
    print(name, "HF:", tuple(hf_weight.shape),
          "My model:", tuple(my_weight.shape))

c_attn HF: (768, 2304) My model: (2304, 768)
attn.c_proj HF: (768, 768) My model: (768, 768)
mlp.c_fc HF: (768, 3072) My model: (3072, 768)
mlp.c_proj HF: (3072, 768) My model: (768, 3072)


In [6]:
text = "The future of artificial intelligence"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    hf_logits = hf_model(**inputs).logits
    my_logits = my_model(inputs["input_ids"])

max_diff = (hf_logits - my_logits).abs().max().item()

print("Hugging Face logits shape:", tuple(hf_logits.shape))
print("My GPT-2 logits shape:", tuple(my_logits.shape))
print("Maximum absolute difference:", max_diff)
print("Outputs match:", torch.allclose(hf_logits, my_logits, atol=1e-4, rtol=1e-4))

Hugging Face logits shape: (1, 5, 50257)
My GPT-2 logits shape: (1, 5, 50257)
Maximum absolute difference: 7.62939453125e-05
Outputs match: True


In [7]:
def generate_top_k(model, tokenizer, prompt, max_new_tokens=40, k=50):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(input_ids) if isinstance(model, MyGPT2) else model(input_ids).logits

        next_token_logits = logits[:, -1, :]

        values, _ = torch.topk(next_token_logits, k)
        threshold = values[:, -1].unsqueeze(-1)

        filtered_logits = next_token_logits.masked_fill(
            next_token_logits < threshold, float("-inf")
        )

        probs = F.softmax(filtered_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [19]:
print(generate_top_k(
    hf_model,
    tokenizer,
    "Artificial intelligence will change the world because",
    max_new_tokens=40,
    k=50
))

turkish_prompt = "Yapay zekânın gelecekte insanlara sağlayacağı en büyük fayda"
print(generate_top_k(
    hf_model,
    tokenizer,
    turkish_prompt,
    max_new_tokens=40,
    k=50
))

turkish_text = "Yapay zekâ gelecekte hayatımızı değiştirecek."
english_text = "Artificial intelligence will change our lives in the future."

turkish_tokens = tokenizer.encode(turkish_text)
english_tokens = tokenizer.encode(english_text)

print("\nTürkçe cümle:", turkish_text)
print("Token sayısı:", len(turkish_tokens))
print("Token parçaları:", tokenizer.convert_ids_to_tokens(turkish_tokens))

print("\nİngilizce cümle:", english_text)
print("Token sayısı:", len(english_tokens))
print("Token parçaları:", tokenizer.convert_ids_to_tokens(english_tokens))

print("\nKarşılaştırma:")
print(f"Türkçe: {len(turkish_tokens)} token | İngilizce: {len(english_tokens)} token")

Artificial intelligence will change the world because it will change the business model of a business.

"Our goal is to build a technology that allows us to be productive, productive and productive in different areas of our life," said Scott B
Yapay zekânın gelecekte insanlara sağlayacağı en büyük faydağı yamıyınyır albınıl vejçte kulat metirın aynnım al büyıy u

Türkçe cümle: Yapay zekâ gelecekte hayatımızı değiştirecek.
Token sayısı: 27
Token parçaları: ['Y', 'ap', 'ay', 'Ġz', 'ek', 'Ã¢', 'Ġge', 'le', 'ce', 'k', 'te', 'Ġhay', 'at', 'Ä±', 'm', 'Ä±', 'z', 'Ä±', 'Ġde', 'ÄŁ', 'i', 'ÅŁ', 't', 'ire', 'ce', 'k', '.']

İngilizce cümle: Artificial intelligence will change our lives in the future.
Token sayısı: 11
Token parçaları: ['Art', 'ificial', 'Ġintelligence', 'Ġwill', 'Ġchange', 'Ġour', 'Ġlives', 'Ġin', 'Ġthe', 'Ġfuture', '.']

Karşılaştırma:
Türkçe: 27 token | İngilizce: 11 token


In [20]:
print("HF model:")
print(generate_top_k(
    hf_model,
    tokenizer,
    "Artificial intelligence will change the world because",
    max_new_tokens=40,
    k=50
))

print("\nBenim modelim:")
print(generate_top_k(
    my_model,
    tokenizer,
    "Artificial intelligence will change the world because",
    max_new_tokens=40,
    k=50
))

HF model:
Artificial intelligence will change the world because of the work of humans on its own. The "smart-phone", which is just a standard computer, will have a huge impact on all parts of its life cycle and that will have a huge

Benim modelim:
Artificial intelligence will change the world because of it," said H. David Zumman, one of the authors of a paper describing the research, published Sunday in the journal Proceedings of the National Academy of Sciences.

The research is


In [9]:
def _init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

sample_text = "Machine learning is changing the world."
input_ids = tokenizer.encode(sample_text, return_tensors="pt").to(device)
initialized_model = MyGPT2(config).to(device)
initialized_model.apply(_init_weights)
initialized_model.eval()
with torch.no_grad():
    logits_init = initialized_model(input_ids)

shift_logits_init = logits_init[:, :-1, :].contiguous()
shift_labels_init = input_ids[:, 1:].contiguous()

loss_init = F.cross_entropy(
    shift_logits_init.view(-1, shift_logits_init.size(-1)),
    shift_labels_init.view(-1)
)

print("Initial loss:", loss_init.item())
print("ln(50257):", torch.log(torch.tensor(50257.0)).item())

Initial loss: 10.826294898986816
ln(50257): 10.824905395507812


In [17]:
random_model = MyGPT2(config).to(device)
random_model.apply(_init_weights)
sample_text = "Machine learning is changing the world."
input_ids = tokenizer.encode(sample_text, return_tensors="pt").to(device)
random_model.train()

optimizer = torch.optim.AdamW(random_model.parameters(), lr=1e-4)

for step in range(300):
    optimizer.zero_grad()

    logits = random_model(input_ids)

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.6f}")

Step   0 | Loss: 10.594226
Step  25 | Loss: 0.016554
Step  50 | Loss: 0.008787
Step  75 | Loss: 0.006751
Step 100 | Loss: 0.005687
Step 125 | Loss: 0.004950
Step 150 | Loss: 0.004410
Step 175 | Loss: 0.004022
Step 200 | Loss: 0.003641
Step 225 | Loss: 0.003344
Step 250 | Loss: 0.003061
Step 275 | Loss: 0.002837


In [18]:
random_model.eval()

with torch.no_grad():
    logits = random_model(input_ids)

    loss_after = F.cross_entropy(
        logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
        input_ids[:, 1:].contiguous().view(-1)
    )

print("Loss after training:", loss_after.item())

Loss after training: 0.002592014614492655
